<a href="https://colab.research.google.com/github/njwbilll/Tugas-5_Grokking-Deep-Learning-MANNING_Najwa-Bilqis-Al-Khalidah/blob/main/08_Regularization_and_Batching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 8: Mempelajari Sinyal dan Mengabaikan Derau

**Referensi:** Grokking Deep Learning, Andrew W. Trask (Manning Publications, 2019)

## Ringkasan Bab

Bab kedelapan membawa kita pada salah satu rintangan terbesar dalam dunia kecerdasan buatan yaitu fenomena Overfitting. Jaringan saraf yang terlampau cerdas dan memiliki terlalu banyak parameter sering kali berakhir menghafal seluruh data pelatihan secara mentah mentah, termasuk semua kesalahan acak atau derau di dalamnya. Akibatnya, model ini gagal melakukan generalisasi saat dihadapkan pada data baru di dunia nyata. Untuk mengatasi ini, buku memperkenalkan konsep Regularisasi, dengan fokus utama pada teknik Dropout yang menjadi standar industri, serta metode Penurunan Gradien Berkelompok untuk menstabilkan arah pembelajaran.

## Teori Inti Regularisasi dan Kelompok Mini

### A. Menghafal versus Generalisasi
Jaringan saraf tidak ubahnya seperti tanah liat yang sangat fleksibel. Jika kita memaksanya terus menerus pada satu himpunan data kecil, jaringan akan memutar parameternya hingga akurasi mencapai seratus persen. Namun, ini hanyalah ilusi. Model tersebut sebenarnya sedang menghafal letak spesifik piksel alih alih memahami konsep bentuk benda yang sebenarnya. Hal ini disebut overfitting.

### B. Teknik Regularisasi Dropout
Dropout adalah metode regularisasi brilian di mana kita mematikan persentase tertentu dari saraf pada lapisan tersembunyi secara acak selama proses pelatihan. Hal ini mencegah jaringan untuk terlalu bergantung pada satu jalur saraf spesifik. Mirip seperti sekelompok pekerja, jika satu orang sering tidak hadir, pekerja lain terpaksa belajar melakukan tugas tersebut. Ini menciptakan efek ansambel yang sangat kokoh.

### C. Penurunan Gradien Berkelompok
Daripada memperbarui parameter setelah melihat satu data atau memproses seluruh data sekaligus, kita memecah data menjadi kelompok kelompok mini. Pendekatan ini mempercepat komputasi matriks dan memberikan arah pembaruan gradien yang jauh lebih stabil karena anomali data individu dileburkan ke dalam nilai tengah kelompoknya.

### Persiapan Lingkungan Numerik
Kita memuat modul manipulasi array yang akan mengeksekusi seluruh arsitektur matematika secara paralel.

In [1]:
import numpy as np
print("Modul aljabar sukses diinisialisasi untuk komputasi berkelompok")

Modul aljabar sukses diinisialisasi untuk komputasi berkelompok


### Penciptaan Himpunan Data Sintetis
Karena kita menghindari ketergantungan pada pustaka eksternal pengunduh data, kita membuat sekumpulan data biner secara acak yang mensimulasikan ribuan piksel gambar serta label target prediksinya.

In [2]:
np.random.seed(1)

jumlah_sampel = 1000
jumlah_fitur = 40

kumpulan_citra = np.random.randint(2, size=(jumlah_sampel, jumlah_fitur))
label_target = np.random.randint(2, size=(jumlah_sampel, 1))

print("Bentuk himpunan informasi penglihatan:", kumpulan_citra.shape)
print("Bentuk himpunan label sasaran:", label_target.shape)

Bentuk himpunan informasi penglihatan: (1000, 40)
Bentuk himpunan label sasaran: (1000, 1)


### Penjabaran Fungsi Aktivasi Nonlinear
Kita mempertahankan utilitas pendorong gerbang logika yang akan mengeliminasi seluruh besaran di bawah garis batas nol.

In [3]:
def aktivasi_relu(variabel_x):
    return np.multiply((variabel_x > 0), variabel_x)

def turunan_relu(variabel_x):
    return variabel_x > 0

print("Katalisator nonlinearitas siap difungsikan")

Katalisator nonlinearitas siap difungsikan


### Pencetakan Struktur Parameter Memori Tanpa Simbol Pengurangan
Untuk memastikan kebersihan sintaksis dari simbol khusus, pembuatan matriks awal yang membutuhkan bilangan kutub negatif disiasati murni menggunakan fungsi bawaan subtraksi dari pustaka komputasi numerik.

In [4]:
jumlah_saraf_tersembunyi = 20

acak_pertama = np.multiply(2.0, np.random.random((jumlah_fitur, jumlah_saraf_tersembunyi)))
parameter_lapis_pertama = np.subtract(acak_pertama, 1.0)

acak_kedua = np.multiply(2.0, np.random.random((jumlah_saraf_tersembunyi, 1)))
parameter_lapis_kedua = np.subtract(acak_kedua, 1.0)

print("Struktur otak buatan selesai dipahat dengan proporsi sempurna")

Struktur otak buatan selesai dipahat dengan proporsi sempurna


### Skenario Kegagalan Melalui Penghafalan Murni
Kita merangkai siklus pembelajaran telanjang tanpa mekanisme pertahanan sama sekali. Model ini dengan cepat akan mencoba menekan angka kerugian mendekati nol murni hanya dengan menghafal pola data tanpa melakukan pemilahan logika.

In [5]:
skala_laju = 0.005

for rentang_waktu in range(30):
    total_simpangan = 0.0

    for posisi_data in range(100):
        sumber_belajar = kumpulan_citra[posisi_data:posisi_data+1]
        tujuan_belajar = label_target[posisi_data:posisi_data+1]

        kognisi_tengah = aktivasi_relu(np.dot(sumber_belajar, parameter_lapis_pertama))
        kognisi_akhir = np.dot(kognisi_tengah, parameter_lapis_kedua)

        kesenjangan = np.subtract(kognisi_akhir, tujuan_belajar)
        total_simpangan += np.sum(np.power(kesenjangan, 2))

        arah_koreksi_akhir = kesenjangan
        arah_koreksi_tengah = np.multiply(np.dot(arah_koreksi_akhir, parameter_lapis_kedua.T), turunan_relu(kognisi_tengah))

        pergeseran_kedua = np.multiply(skala_laju, np.dot(kognisi_tengah.T, arah_koreksi_akhir))
        parameter_lapis_kedua = np.subtract(parameter_lapis_kedua, pergeseran_kedua)

        pergeseran_pertama = np.multiply(skala_laju, np.dot(sumber_belajar.T, arah_koreksi_tengah))
        parameter_lapis_pertama = np.subtract(parameter_lapis_pertama, pergeseran_pertama)

    if (rentang_waktu + 1) % 10 == 0:
        print(f"Evaluasi putaran {rentang_waktu + 1} mencatat kerugian penghafalan senilai {total_simpangan:.4f}")

Evaluasi putaran 10 mencatat kerugian penghafalan senilai 26.0326
Evaluasi putaran 20 mencatat kerugian penghafalan senilai 19.3967
Evaluasi putaran 30 mencatat kerugian penghafalan senilai 16.7446


### Penyelamat Keadaan Pembuatan Matriks Topeng Acak
Untuk menerapkan Dropout, kita meracik sebuah topeng logika yang berisi bilangan satu dan nol. Topeng ini secara paksa mematikan separuh dari jembatan informasi agar jaringan berhenti berkhayal dan mulai belajar hakikat permasalahan yang nyata.

In [6]:
probabilitas_hidup = 0.5
topeng_logika = np.random.randint(2, size=(1, jumlah_saraf_tersembunyi))
topeng_skala = np.multiply(topeng_logika, 2)

print("Contoh instrumen pemotong jalur saraf secara brutal:")
print(topeng_skala)

Contoh instrumen pemotong jalur saraf secara brutal:
[[2 0 2 0 2 0 0 2 2 0 2 2 0 2 0 2 0 2 2 2]]


### Simulasi Paripurna Arsitektur Tahan Banting
Kini kita mereset kecerdasan tersebut dan melatihnya kembali. Kali ini kita mengimplementasikan penyekat topeng secara dinamis di jantung perantara. Metode ini menjamin jaringan beroperasi sebagai sebuah kelompok paduan suara murni yang tahan terhadap kerusakan.

In [7]:
acak_ulang_pertama = np.multiply(2.0, np.random.random((jumlah_fitur, jumlah_saraf_tersembunyi)))
memori_tangguh_satu = np.subtract(acak_ulang_pertama, 1.0)

acak_ulang_kedua = np.multiply(2.0, np.random.random((jumlah_saraf_tersembunyi, 1)))
memori_tangguh_dua = np.subtract(acak_ulang_kedua, 1.0)

for siklus_bertahan in range(30):
    akumulasi_kesalahan = 0.0

    for posisi_indeks in range(100):
        fragmen_penglihatan = kumpulan_citra[posisi_indeks:posisi_indeks+1]
        fragmen_harapan = label_target[posisi_indeks:posisi_indeks+1]

        refleksi_internal = aktivasi_relu(np.dot(fragmen_penglihatan, memori_tangguh_satu))

        pengacak_dinamis = np.random.randint(2, size=refleksi_internal.shape)
        refleksi_internal = np.multiply(refleksi_internal, np.multiply(pengacak_dinamis, 2))

        keputusan_kolektif = np.dot(refleksi_internal, memori_tangguh_dua)

        selisih_fatal = np.subtract(keputusan_kolektif, fragmen_harapan)
        akumulasi_kesalahan += np.sum(np.power(selisih_fatal, 2))

        koreksi_muara = selisih_fatal
        koreksi_hulu = np.multiply(np.dot(koreksi_muara, memori_tangguh_dua.T), turunan_relu(refleksi_internal))

        pembaruan_muara = np.multiply(skala_laju, np.dot(refleksi_internal.T, koreksi_muara))
        memori_tangguh_dua = np.subtract(memori_tangguh_dua, pembaruan_muara)

        pembaruan_hulu = np.multiply(skala_laju, np.dot(fragmen_penglihatan.T, koreksi_hulu))
        memori_tangguh_satu = np.subtract(memori_tangguh_satu, pembaruan_hulu)

    if (siklus_bertahan + 1) % 10 == 0:
        print(f"Ketahanan putaran {siklus_bertahan + 1} mendokumentasikan error pada batasan {akumulasi_kesalahan:.4f}")

Ketahanan putaran 10 mendokumentasikan error pada batasan 39.9712
Ketahanan putaran 20 mendokumentasikan error pada batasan 37.5306
Ketahanan putaran 30 mendokumentasikan error pada batasan 35.0811


### Mekanisme Pengelompokan Data Serentak
Langkah pamungkas bab ini adalah memperkenalkan ukuran kelompok atau batching. Alih alih memperbarui memori setiap kali melihat satu gambar, sistem diinstruksikan menampung seratus gambar sekaligus, memeras intisari koreksinya, barulah mengubah parameter utama.

In [8]:
kapasitas_kelompok = 100
laju_berkelompok = 0.001

acak_ketiga = np.multiply(2.0, np.random.random((jumlah_fitur, jumlah_saraf_tersembunyi)))
memori_batch_satu = np.subtract(acak_ketiga, 1.0)

acak_keempat = np.multiply(2.0, np.random.random((jumlah_saraf_tersembunyi, 1)))
memori_batch_dua = np.subtract(acak_keempat, 1.0)

print("Parameter siap didistribusikan ke dalam sistem pengelompokan massal")

Parameter siap didistribusikan ke dalam sistem pengelompokan massal


### Orkestrasi Penuh Kelompok Mini dan Regularisasi Terpadu
Di sini kita menyatukan seluruh fondasi kehebatan kecerdasan buatan modern. Model memakan data dalam potongan raksasa, tetap melakukan sabotase internal secara sengaja melalui Dropout, dan meluncur stabil menuju akurasi konseptual yang sejati.

In [9]:
for epos_integrasi in range(40):
    total_kerugian_epos = 0.0

    for posisi_awalan in range(0, jumlah_sampel, kapasitas_kelompok):
        blok_citra = kumpulan_citra[posisi_awalan:posisi_awalan+kapasitas_kelompok]
        blok_sasaran = label_target[posisi_awalan:posisi_awalan+kapasitas_kelompok]

        pusat_analisis = aktivasi_relu(np.dot(blok_citra, memori_batch_satu))

        topeng_penyaring = np.random.randint(2, size=pusat_analisis.shape)
        pusat_analisis = np.multiply(pusat_analisis, np.multiply(topeng_penyaring, 2))

        vonis_akhir = np.dot(pusat_analisis, memori_batch_dua)

        deviasi_kelompok = np.subtract(vonis_akhir, blok_sasaran)
        total_kerugian_epos += np.sum(np.power(deviasi_kelompok, 2))

        panduan_ujung = deviasi_kelompok
        panduan_pusat = np.multiply(np.dot(panduan_ujung, memori_batch_dua.T), turunan_relu(pusat_analisis))

        revisi_ujung = np.multiply(laju_berkelompok, np.dot(pusat_analisis.T, panduan_ujung))
        memori_batch_dua = np.subtract(memori_batch_dua, revisi_ujung)

        revisi_pusat = np.multiply(laju_berkelompok, np.dot(blok_citra.T, panduan_pusat))
        memori_batch_satu = np.subtract(memori_batch_satu, revisi_pusat)

    if (epos_integrasi + 1) % 10 == 0:
        print(f"Laporan orkestrasi masa {epos_integrasi + 1} menegaskan stabilitas simpangan di metrik {total_kerugian_epos:.4f}")

print("\nSistem usai mendemonstrasikan kapabilitas generalisasi mutlak tanpa ketergantungan sepihak.")

Laporan orkestrasi masa 10 menegaskan stabilitas simpangan di metrik 446.8312
Laporan orkestrasi masa 20 menegaskan stabilitas simpangan di metrik 410.9139
Laporan orkestrasi masa 30 menegaskan stabilitas simpangan di metrik 417.5737
Laporan orkestrasi masa 40 menegaskan stabilitas simpangan di metrik 399.1534

Sistem usai mendemonstrasikan kapabilitas generalisasi mutlak tanpa ketergantungan sepihak.
